# Важность признаков для предсказания типа экзамена

> Выявим малоинформативные признаки, которые особо ничего не дают нашей модели

In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, f1_score
from catboost import CatBoostRegressor, Pool
from transform_utils import LightGBM_model
from data_utils import fillna_data

In [64]:
df = pd.read_csv('total_laggs.csv')
df = fillna_data(df)

X = df.drop(columns=['target_grade', 'target_type', 'student_id'])
y1 = df['target_grade']
y2 = df['target_type']

In [65]:
X_train, X_test, y1_train, y1_test = train_test_split(X, y1, test_size=0.3) 

In [66]:
CATBOOST_HYPERPARAMETRS = {'iterations': 329, 'learning_rate': 0.20159023821563268, 'loss_function': 'MAE', 'depth': 4, 'l2_leaf_reg': 79.57030243930838, 'min_data_in_leaf': 60}
LIGHTGBM_HYPERPARAMETRS = {'learning_rate': 0.05593308405682298, 'num_leaves': 49, 'max_depth': 5, 'n_estimators': 523, 'bagging_fraction': 0.7553386111732491, 'min_child_samples': 177, 'feature_fraction': 0.5700191961275134}

In [67]:
X_train_, X_test_ = train_test_split(df, test_size=0.3)

In [68]:
type_model = LightGBM_model(**LIGHTGBM_HYPERPARAMETRS)
type_model.fit(X_train_)

In [69]:
y2_pred = type_model.predict_type(X_test_)

test_f1 = f1_score(X_test_['target_type'], y2_pred, average='weighted') 
print('F1-score:', test_f1)

F1-score: 0.959105539439277


In [55]:
columns = X_train.columns

type_imp = type_model.feature_importances_

Важность признаков (в процентах):

In [56]:
m = len(columns)

res = {'feature': columns, 
       'importance for exam type': [type_imp[i] for i in range(m)]}
res = pd.DataFrame(res).sort_values(by=[ 'importance for exam type'], ascending=False)

res

,feature,importance for exam type
19,min_prev,13.027104
10,difficulty_avg_score,10.295444
15,difficulty_prev,9.833979
1,education_level,7.993423
0,program,7.277356
7,module,7.012147
9,grade_10,6.062696
6,discipline,4.614650
11,std_deviation,4.296398
18,avg_grade_prev,3.638678


Cоединим с важностью признаков для оценок: 

In [57]:
grade_ = pd.read_csv('feature_importance_grade.csv')
merged = res.merge(grade_, on='feature')

merged

,feature,importance for exam type,importance for grade
0,min_prev,13.027104,1.096250
1,difficulty_avg_score,10.295444,2.651546
2,difficulty_prev,9.833979,3.834440
3,education_level,7.993423,0.052935
4,program,7.277356,1.519250
5,module,7.012147,1.132251
6,grade_10,6.062696,39.674927
7,discipline,4.614650,3.501773
8,std_deviation,4.296398,3.370498
9,avg_grade_prev,3.638678,4.343455


In [58]:
merged['importance mean'] = merged[['importance for exam type', 'importance for grade']].mean(axis=1)
merged = merged.sort_values('importance mean', ascending=False)

merged.to_csv('feature_importance_mean.csv', index=False)

merged

,feature,importance for exam type,importance for grade,importance mean
6,grade_10,6.062696,39.674927,22.868811
0,min_prev,13.027104,1.096250,7.061677
2,difficulty_prev,9.833979,3.834440,6.834209
1,difficulty_avg_score,10.295444,2.651546,6.473495
14,course,2.498276,10.211883,6.355080
4,program,7.277356,1.519250,4.398303
5,module,7.012147,1.132251,4.072199
7,discipline,4.614650,3.501773,4.058211
3,education_level,7.993423,0.052935,4.023179
9,avg_grade_prev,3.638678,4.343455,3.991066
